# 모델 실험 템플릿

팀 공통 실험 노트북입니다. **이 파일은 고치지 말고 복사해서** 씁니다.
각자 모델 하나(`yolo11m.pt` 또는 `yolo26m.pt`)를 맡아 옵티마이저·lr0 등을 조정합니다.

1. `notebooks/exp_<이름>.ipynb`로 복사합니다.
2. 설정 셀의 **내 실험** 부분만 바꿉니다. **팀 공통** 값은 모두 같아야 서로 점수를 비교할 수 있으므로 바꾸지 않습니다.
3. 순서
   1. 로컬(Mac)에서 `SMOKE = True`로 동작 확인 (epoch 5, 학습 사진 10%, wandb 끔)
   2. Colab GPU에서 `SMOKE = False`, `STAGE = "screen"`: fold 0만 학습해 설정을 고릅니다. 다른 설정과는 fold 0 점수끼리 비교합니다.
   3. 고른 설정은 `STAGE = "confirm"`: 5 fold를 모두 학습합니다. fold 0은 screen 결과를 그대로 씁니다. **모델 비교는 이 5 fold 평균·표준편차로** 합니다.
4. 실험 이름은 설정으로 자동으로 만듭니다 (예: `a_yolo26m_AdamW_lr0.001_cos`). 같은 설정을 다시 돌리면 끝난 fold는 건너뛰고, 끊긴 fold는 `last.pt`에서 이어 학습합니다.
5. 결과는 wandb 팀 프로젝트에 `<실험 이름>/<run>` 이름으로 쌓이고, config에 옵티마이저·lr0 등이 남습니다.
   - Colab: 커널 선택 → **Colab** → GPU 런타임. VS Code 탐색기에서 `.env`를 우클릭 → **Upload to Colab** (`DATA_URL`, `WANDB_API_KEY`, `WANDB_ENTITY`).
   - Colab은 `BRANCH`의 코드를 받아 돌리므로, 로컬에서 고친 `.py`는 push한 뒤 실행합니다.
6. 커밋할 때는 출력을 지웁니다.

## 1. 설정

실험마다 **내 실험**과 **실행** 부분만 바꿉니다. 팀 공통 값을 바꿔야 할 일이 생기면 팀에서 정한 뒤 템플릿을 고칩니다.

In [ ]:
# ===== 팀 공통: 바꾸지 않는다 (AGENTS.md 기본값) =====
LABEL = "v1"
SPLIT = "setup30fix_v1_s42"
EPOCHS = 30  # 이번 테스트는 30에폭. 팀 기본값은 100
IMGSZ = 640
SEED = 42
AUG = "on"  # models/train.py의 AUG_PRESETS
FRACTION = 1.0

# ===== 내 실험 =====
MEMBER = "JJAM99"  # 이름이나 이니셜. 실험 이름 맨 앞에 붙는다
MODEL = "yolo11m.pt"  # yolo11m.pt 또는 yolo26m.pt
# SGD, MuSGD, Adam, Adamax, AdamW, NAdam, RAdam, RMSProp. auto는 lr0를 무시하므로 쓰지 않는다
OPTIMIZER = "MuSGD"
LR0 = None  # None이면 SGD 계열 0.01, 그 밖 0.001
COS_LR = False  # True면 학습률을 cosine 곡선으로 줄인다
BATCH = 16  # 팀 기본 16. GPU 메모리가 부족하면 줄인다
# 그 밖 train() 인자. 괄호 안은 ultralytics 기본값
# lrf(0.01), momentum(0.937), weight_decay(0.0005), warmup_epochs(3.0), patience(100)
TRAIN_KWARGS = {}  # 예: {"weight_decay": 0.001, "patience": 30}
TAG = ""  # 이름이 같아지는 실험을 구분할 짧은 꼬리표 (선택)

# ===== 실행 =====
SMOKE = False  # True면 동작 확인만. Colab GPU 학습은 False
STAGE = "screen"  # screen: fold 0만, confirm: 5 fold 전부
BRANCH = "main"  # Colab에서 받을 브랜치

## 2. 환경

Colab이면 저장소를 받고 패키지를 설치한 뒤 결과 폴더를 Drive에 둡니다. 로컬에서는 프로젝트 루트를 import 경로에 넣기만 합니다.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").is_dir()
RUNS_DIR = None  # None이면 프로젝트의 runs/

if IN_COLAB:
    # colab.ipynb 2–4단계와 같다: 저장소 받기, uv.lock 버전 설치, .env 읽기
    import importlib

    REPO_DIR = Path("/content/pill-bbox")
    if REPO_DIR.exists():
        !git -C {REPO_DIR} fetch -q origin {BRANCH}
        !git -C {REPO_DIR} checkout -q {BRANCH}
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q -b {BRANCH} https://github.com/def-uk/pill-bbox.git {REPO_DIR}
    os.chdir(REPO_DIR)
    if not Path("~/.local/bin/uv").expanduser().exists():
        !curl -LsSf https://astral.sh/uv/install.sh | sh
    os.environ["PATH"] = f"{Path('~/.local/bin').expanduser()}:{os.environ['PATH']}"
    # torch는 lock에 적힌 CUDA 12.6 빌드로 설치한다
    req = "/tmp/requirements.txt"
    torch_index = "--extra-index-url https://download.pytorch.org/whl/cu126"
    strategy = "--index-strategy unsafe-best-match"
    !uv export -q --frozen --no-dev --no-hashes --no-emit-project -o {req}
    !uv pip install -q --python {sys.executable} -r {req} {torch_index} {strategy}
    sys.path.insert(0, str(REPO_DIR / "notebooks"))
    importlib.invalidate_caches()

    from dotenv import load_dotenv

    load_dotenv("/content/.env")  # 이미 설정된 환경 변수가 우선한다
    for name in ["DATA_URL", "WANDB_API_KEY", "WANDB_ENTITY"]:
        if not os.getenv(name):
            try:
                from google.colab import userdata

                os.environ[name] = userdata.get(name)
            except Exception:
                pass
    if not os.getenv("WANDB_API_KEY"):
        os.environ["WANDB_MODE"] = "disabled"
        print("WANDB_API_KEY가 없어 wandb를 끄고 진행합니다.")

    # 결과를 Drive에 두어야 런타임이 끊겨도 남는다
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        RUNS_DIR = Path("/content/drive/MyDrive/pill-bbox/runs")
    except Exception as e:
        print(f"Drive 연결 실패({e}). 결과가 런타임에만 남으니 끊기기 전에 내려받으세요.")
    !git branch --show-current && git log --oneline -1

# 프로젝트 루트를 import 경로에 넣는다
import nb_data  # noqa: E402, F401

## 3. 실험 준비

설정을 검사하고 실험 이름을 만듭니다. 같은 학습을 터미널에서 다시 돌리는 `model_main` 명령을 출력하고, 설정은 `<RUNS_DIR>/<exp>/notebook_config.json`에 남깁니다.

In [ ]:
import json
from datetime import datetime

from data_process.split import N_FOLDS
from models.train import default_lr0
from src.config import PROJECT_ROOT, get_device
from src.config import RUNS_DIR as DEFAULT_RUNS_DIR

# 가중치 이름(yolo11m.pt)은 현재 폴더에 받아지므로 루트로 옮겨 팀 공통 위치에 둔다
os.chdir(PROJECT_ROOT)
RUNS_DIR = RUNS_DIR or DEFAULT_RUNS_DIR
DEVICE = get_device()

if not MEMBER:
    raise ValueError("MEMBER에 이름이나 이니셜을 적으세요.")
if MODEL not in ["yolo11m.pt", "yolo26m.pt"]:
    raise ValueError(f"후보 모델은 yolo11m.pt, yolo26m.pt입니다: {MODEL}")
if OPTIMIZER == "auto":
    raise ValueError("auto는 lr0를 무시합니다. 옵티마이저를 직접 고르세요 (예: AdamW).")
if STAGE not in ["screen", "confirm"]:
    raise ValueError(f"STAGE는 screen 또는 confirm입니다: {STAGE}")
lr0 = default_lr0(OPTIMIZER) if LR0 is None else LR0

# 결과 폴더 이름에는 lr0 등이 없으므로, 내 실험 설정을 모두 실험 이름에 넣어 구분한다
parts = [MEMBER, Path(MODEL).stem, OPTIMIZER, f"lr{lr0:g}"]
parts += ["cos"] if COS_LR else []
parts += [] if BATCH == 16 else [f"b{BATCH}"]
parts += [
    f"{key}{value:g}" if isinstance(value, float) else f"{key}{value}"
    for key, value in sorted(TRAIN_KWARGS.items())
]
parts += [TAG] if TAG else []
exp_name = "_".join(parts)

# 동작 확인이면 짧은 값을 쓰고, 실험 이름에 시각을 붙여 이전 확인과 겹치지 않게 한다
if SMOKE:
    exp, epochs, fraction, folds = (
        f"smoke_{exp_name}_{datetime.now():%m%d_%H%M}",
        5,
        0.1,
        [0],
    )
    os.environ["WANDB_MODE"] = "disabled"
else:
    if DEVICE != "cuda":
        raise RuntimeError(f"본 학습은 GPU에서 합니다(지금 {DEVICE}). 동작 확인은 SMOKE = True.")
    exp = f"{exp_name}_test30" if EPOCHS == 30 else exp_name
    epochs, fraction = EPOCHS, FRACTION
    folds = [0] if STAGE == "screen" else list(range(N_FOLDS))

# 같은 학습을 터미널에서 다시 돌릴 수 있게 model_main 명령으로 남긴다
cmd = ["uv run python -m models.model_main", "--model", MODEL, "--label", LABEL]
cmd += [
    "--split",
    SPLIT,
    "--folds",
    *map(str, folds),
    "--epochs",
    str(epochs),
    "--exp",
    exp,
]
cmd += ["--imgsz", str(IMGSZ), "--batch", str(BATCH), "--seed", str(SEED)]
cmd += [
    "--fraction",
    str(fraction),
    "--aug",
    AUG,
    "--optimizer",
    OPTIMIZER,
    "--lr0",
    f"{lr0:g}",
]
cmd += ["--cos-lr"] if COS_LR else []
cmd += [] if RUNS_DIR == DEFAULT_RUNS_DIR else ["--runs-dir", str(RUNS_DIR)]
cmd += ["--resume"]
command = " ".join(cmd)

config = {
    "exp": exp,
    "member": MEMBER,
    "stage": STAGE,
    "smoke": SMOKE,
    "device": DEVICE,
    "model": MODEL,
    "label": LABEL,
    "split": SPLIT,
    "folds": folds,
    "epochs": epochs,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "fraction": fraction,
    "aug": AUG,
    "optimizer": OPTIMIZER,
    "lr0": lr0,
    "cos_lr": COS_LR,
    "train_kwargs": TRAIN_KWARGS,
    "command": command,
    "created_at": datetime.now().isoformat(timespec="seconds"),
}
exp_dir = RUNS_DIR / exp
exp_dir.mkdir(parents=True, exist_ok=True)
(exp_dir / "notebook_config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"실험 {exp} ({STAGE}, fold {folds}), 장치 {DEVICE}")
print(f"결과 {exp_dir}")
print(command)
if TRAIN_KWARGS:
    print(f"추가 train() 인자 (CLI에 없음): {TRAIN_KWARGS}")

## 4. 데이터

데이터는 `data_process.data_main`으로만 만듭니다. fold마다 train/val 사진 수입니다.

In [ ]:
from nb_data import ensure_pipeline, load_folds

# 라벨·fold·data.yaml이 없으면 data_main을 실행해 만든다 (처음 약 25초, 원본이 없으면 받기부터)
ensure_pipeline(LABEL, [SPLIT])
load_folds(SPLIT).groupby(["fold", "split"]).size().unstack()

In [ ]:
# 평가·비교 함수: 보조 파일 없이 이 노트북 안에서 함께 사용한다.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from matplotlib.ticker import MaxNLocator
from ultralytics import YOLO
from ultralytics.data.utils import img2label_paths
from ultralytics.utils.metrics import box_iou
from ultralytics.utils.ops import xywh2xyxy

from data_process.data_yaml import data_yaml_path
from models.evaluate import map75_95
from models.train import MAIN_METRIC_KEY
from src.config import get_device


def matched_iou(model: YOLO, data: Path, imgsz: int, batch: int) -> dict[str, float | int]:
    """같은 클래스·IoU 0.5 이상인 예측과 정답을 일대일 매칭해 평균과 개수를 반환한다."""
    dataset = yaml.safe_load(data.read_text(encoding="utf-8"))
    matched = []
    target_count = 0
    prediction_count = 0
    for prediction in model.predict(
        source=dataset["val"],
        imgsz=imgsz,
        batch=batch,
        device=get_device(),
        conf=0.001,
        iou=0.7,
        stream=True,
        verbose=False,
        save=False,
    ):
        label_path = Path(img2label_paths([prediction.path])[0])
        targets = np.array(
            [
                list(map(float, line.split()))
                for line in label_path.read_text().splitlines()
                if line.strip()
            ],
            dtype=np.float32,
        ).reshape(-1, 5)
        target_count += len(targets)
        prediction_count += len(prediction.boxes)
        if not len(targets) or not len(prediction.boxes):
            continue
        height, width = prediction.orig_shape
        boxes = xywh2xyxy(torch.from_numpy(targets[:, 1:].copy()))
        boxes *= torch.tensor([width, height, width, height])
        overlaps = box_iou(boxes, prediction.boxes.xyxy.cpu()).numpy()
        same_class = targets[:, 0, None] == prediction.boxes.cls.cpu().numpy()[None, :]
        target_indices, prediction_indices = np.where(same_class & (overlaps >= 0.5))
        order = np.argsort(-overlaps[target_indices, prediction_indices], kind="stable")
        used_targets, used_predictions = set(), set()
        for index in order:
            target_index = int(target_indices[index])
            prediction_index = int(prediction_indices[index])
            if target_index in used_targets or prediction_index in used_predictions:
                continue
            used_targets.add(target_index)
            used_predictions.add(prediction_index)
            matched.append(float(overlaps[target_index, prediction_index]))
    return {
        "Mean IoU": float(np.mean(matched)) if matched else float("nan"),
        "Matched": len(matched),
        "Targets": target_count,
        "Unmatched targets": target_count - len(matched),
        "Predictions": prediction_count,
    }


def find_smoke_runs(reference: Path) -> list[Path]:
    """같은 모델·분할·fold와 주요 설정으로 완료된 5에폭 이하 스모크를 찾는다."""
    settings = yaml.safe_load((reference / "args.yaml").read_text())
    keys = (
        "model",
        "optimizer",
        "lr0",
        "cos_lr",
        "batch",
        "imgsz",
        "seed",
        "degrees",
        "fliplr",
        "flipud",
        "hsv_h",
        "hsv_s",
        "hsv_v",
        "mosaic",
        "momentum",
        "weight_decay",
        "warmup_epochs",
        "patience",
        "lrf",
        "freeze",
    )
    candidates = []
    for path in reference.parent.parent.glob(f"smoke_*/{reference.name}"):
        args_path = path / "args.yaml"
        history_path = path / "results.csv"
        if not args_path.is_file() or not history_path.is_file():
            continue
        args = yaml.safe_load(args_path.read_text())
        if not (path / "weights" / "best.pt").is_file():
            continue
        if not 0 < args["epochs"] <= 5 or args.get("fraction") != 0.1:
            continue
        if any(args.get(key) != settings.get(key) for key in keys):
            continue
        history = pd.read_csv(history_path)
        history.columns = history.columns.str.strip()
        if len(history) and history["epoch"].max() >= args["epochs"]:
            candidates.append(path)
    return sorted(candidates, key=lambda path: (path / "weights" / "best.pt").stat().st_mtime)


def compare_runs(
    runs: dict[str, Path],
    label: str,
    split: str,
    fold: int,
    imgsz: int,
    batch: int,
    output_dir: Path,
) -> pd.DataFrame:
    """각 run의 best.pt를 같은 val 전체로 평가해 비교표·최종 지표 그림·학습 곡선을 저장한다."""
    output_dir.mkdir(parents=True, exist_ok=True)
    data = data_yaml_path(label, split, fold)
    rows = []
    histories = {}
    for name, run in runs.items():
        settings = yaml.safe_load((run / "args.yaml").read_text())
        history = pd.read_csv(run / "results.csv")
        history.columns = history.columns.str.strip()
        histories[name] = history
        weights = run / "weights" / "best.pt"
        print(f"{name}: {weights}")
        model = YOLO(str(weights))
        result = model.val(
            data=str(data),
            imgsz=imgsz,
            batch=batch,
            device=get_device(),
            conf=0.001,
            iou=0.7,
            plots=False,
            project=str(output_dir / "val"),
            name=name,
            exist_ok=True,
        )
        rows.append(
            {
                "Experiment": name,
                "Epochs": int(history["epoch"].max()),
                "Fraction": settings["fraction"],
                "mAP75-95": map75_95(result.box),
                "Recall": float(result.box.mr),
                "Precision": float(result.box.mp),
                **matched_iou(model, data, imgsz, batch),
                "Weights": str(weights),
            }
        )
    summary = pd.DataFrame(rows).set_index("Experiment")
    columns = [
        "Epochs",
        "Fraction",
        "mAP75-95",
        "Mean IoU",
        "Recall",
        "Precision",
        "Matched",
        "Targets",
        "Unmatched targets",
        "Predictions",
        "Weights",
    ]
    summary = summary[columns]
    summary.to_csv(output_dir / "comparison.csv", encoding="utf-8")
    _plot_comparison(summary, histories, output_dir)
    return summary


def _plot_comparison(
    summary: pd.DataFrame,
    histories: dict[str, pd.DataFrame],
    output_dir: Path,
) -> None:
    """최종 지표의 막대그래프와 에폭별 꺾은선 그래프를 표시하고 저장한다."""
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    for ax, metric in zip(axes, ["mAP75-95", "Mean IoU", "Recall", "Precision"], strict=True):
        values = summary[metric]
        bars = ax.bar(summary.index, values, color=[f"C{i}" for i in range(len(summary))])
        ax.bar_label(bars, labels=[f"{value:.4f}" for value in values], padding=3)
        ax.set(title=metric, ylim=(0, 1.1))
        ax.tick_params(axis="x", labelrotation=20)
        ax.grid(axis="y", alpha=0.3)
    fig.suptitle("Best checkpoint: same validation set (IoU: matched objects only)")
    fig.tight_layout()
    fig.savefig(output_dir / "comparison_metrics.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for ax, (key, title) in zip(
        axes.flat,
        [
            (MAIN_METRIC_KEY, "Validation mAP75-95"),
            ("metrics/recall(B)", "Validation recall"),
            ("metrics/precision(B)", "Validation precision"),
            ("train/cls_loss", "Classification loss (solid: train / dashed: val)"),
        ],
        strict=True,
    ):
        for index, (name, history) in enumerate(histories.items()):
            ax.plot(history["epoch"], history[key], marker=".", color=f"C{index}", label=name)
            if key == "train/cls_loss":
                ax.plot(history["epoch"], history["val/cls_loss"], "--", color=f"C{index}")
        ax.set(title=title, xlabel="Epoch")
        if key != "train/cls_loss":
            ax.set_ylim(0, 1)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle("Training history: smoke 10% / test 100% (data and epochs differ)")
    fig.tight_layout()
    fig.savefig(output_dir / "comparison_history.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 5. 학습과 평가

`best.pt`는 검증 mAP75-95가 가장 높은 epoch입니다. 런타임이 끊기면 위에서부터 다시 실행하면 이어서 학습합니다.

In [ ]:
from models.evaluate import MAIN_METRIC, evaluate_fold, summarize_exp
from models.train import train

# 실험 이름이 설정에서 나오므로 같은 이름 = 같은 설정이다. 그래서 항상 resume으로 돌린다:
# 끝난 fold(screen의 fold 0 등)는 건너뛰고, 끊긴 fold는 last.pt에서 이어 간다.
for fold in folds:
    save_dir = train(
        MODEL,
        LABEL,
        SPLIT,
        fold,
        epochs=epochs,
        exp=exp,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        fraction=fraction,
        aug=AUG,
        optimizer=OPTIMIZER,
        lr0=lr0,
        cos_lr=COS_LR,
        runs_dir=RUNS_DIR,
        resume=True,
        **TRAIN_KWARGS,
    )
    _, summary = evaluate_fold(
        save_dir / "weights" / "best.pt",
        LABEL,
        SPLIT,
        fold,
        exp=exp,
        imgsz=IMGSZ,
        batch=BATCH,
        runs_dir=RUNS_DIR,
    )
    print(f"{save_dir.name}: {MAIN_METRIC} {summary[MAIN_METRIC]:.4f}")

## 6. 결과 요약

다른 사람·다른 설정과 비교할 때 screen은 fold 0끼리, 모델 비교는 confirm의 5 fold 평균·표준편차로 합니다.

In [ ]:
# fold 평균·표준편차. <exp>/results/summary.csv에도 저장된다. screen은 fold 하나라 std가 비어 있다.
summary = summarize_exp(exp, RUNS_DIR)
cols = [("n_folds", "count"), (MAIN_METRIC, "mean"), (MAIN_METRIC, "std")]
cols += [("map50", "mean"), ("map50_95", "mean")]
summary[cols].round(4)

## 7. 학습 곡선

fold별 매 epoch 학습·검증 loss(윗줄)와 검증 mAP(아랫줄)입니다. 같은 그림이 run 폴더의 `results.png`로도 저장됩니다.

In [ ]:
import pandas as pd

METRIC_KEYS = ["metrics/mAP50(B)", "metrics/mAP50-95(B)", MAIN_METRIC_KEY]

# ultralytics가 매 epoch 쓰는 results.csv를 fold별로 읽는다 (run 폴더 이름 끝이 _fold<k>)
histories = {}
for path in sorted(exp_dir.glob("*/results.csv")):
    history = pd.read_csv(path)
    history.columns = history.columns.str.strip()
    histories[path.parent.name.rsplit("_", 1)[-1]] = history

if not histories:
    print(f"{exp_dir}에 results.csv가 없습니다. 학습 셀을 먼저 실행하세요.")
else:
    first = next(iter(histories.values()))
    loss_names = [c.removeprefix("train/") for c in first.columns if c.startswith("train/")]
    n_cols = max(len(loss_names), len(METRIC_KEYS))
    fig, axes = plt.subplots(2, n_cols, figsize=(4.5 * n_cols, 7), squeeze=False)
    for i, (fold, history) in enumerate(histories.items()):
        color, epoch = f"C{i}", history["epoch"]
        # 윗줄: loss. 실선은 train, 점선은 val. val이 오르는데 train만 내려가면 과적합이다
        for ax, name in zip(axes[0], loss_names, strict=False):
            ax.plot(
                epoch,
                history[f"train/{name}"],
                color=color,
                marker=".",
                label=f"{fold} train",
            )
            ax.plot(
                epoch,
                history[f"val/{name}"],
                color=color,
                linestyle="--",
                label=f"{fold} val",
            )
            ax.set_title(name)
        # 아랫줄: 검증 mAP. 별은 best.pt를 고른 epoch (mAP75-95 최고)
        for ax, key in zip(axes[1], METRIC_KEYS, strict=False):
            ax.plot(epoch, history[key], color=color, marker=".", label=fold)
            ax.set_title(key.removeprefix("metrics/").removesuffix("(B)"))
        best = history[MAIN_METRIC_KEY].idxmax()
        axes[1, len(METRIC_KEYS) - 1].plot(
            epoch[best],
            history.loc[best, MAIN_METRIC_KEY],
            "*",
            color=color,
            markersize=14,
        )
    for ax in axes.ravel():
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    for ax in [*axes[0, len(loss_names) :], *axes[1, len(METRIC_KEYS) :]]:
        ax.set_visible(False)
    axes[0, 0].legend(fontsize=7)
    axes[1, 0].legend(fontsize=7)
    fig.suptitle(exp)
    fig.tight_layout()
    plt.show()

In [ ]:
# 8. AP: best.pt를 val 전체로 평가한다. 이 셀은 학습 없이 검증 추론을 실행한다.
# AP는 클래스별 값, mAP는 클래스 평균이다. 주 지표 mAP75-95는 그대로 유지한다.
import pandas as pd
from IPython.display import display
from ultralytics import YOLO

from data_process.data_yaml import load_class_map
from models.evaluate import ap75_95

metric_runs = []
class_map = load_class_map(LABEL)
for weights in sorted(exp_dir.glob("*/weights/best.pt")):
    metric_fold = int(weights.parent.parent.name.rsplit("_fold", 1)[1])
    if metric_fold not in folds:
        continue
    metric_model = YOLO(str(weights))
    metric_data = data_yaml_path(LABEL, SPLIT, metric_fold)
    metric_result = metric_model.val(
        data=str(metric_data),
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        conf=0.001,
        iou=0.7,
        plots=False,
        project=str(exp_dir / "results" / "metric_val"),
        name=weights.parent.parent.name,
        exist_ok=True,
    )
    metric_runs.append((weights.parent.parent.name, metric_model, metric_data, metric_result))

if not metric_runs:
    raise FileNotFoundError("선택한 fold의 best.pt가 없습니다. 학습 셀을 먼저 실행하세요.")

ap_rows = []
for run_name, _, _, result in metric_runs:
    box = result.box
    class_ap = pd.DataFrame(
        {
            "yolo_idx": box.ap_class_index,
            "AP50": box.ap50,
            "AP50-95": box.ap,
            "AP75-95": ap75_95(box.all_ap),
        }
    ).merge(class_map, on="yolo_idx", how="left")
    print(run_name, "클래스별 AP (val에 정답이 있는 클래스)")
    display(class_ap.round(4))
    ap_rows.append(
        {
            "run": run_name,
            "mAP50": box.map50,
            "mAP50-95": box.map,
            "mAP75-95": map75_95(box),
        }
    )
display(pd.DataFrame(ap_rows).set_index("run").round(4))

In [ ]:
# 9. IoU: 같은 클래스·IoU >= 0.5인 예측과 정답을 일대일 매칭한다.
# 30에폭 비교 셀과 같은 계산 함수를 사용한다. 미검출은 평균에서 빠진다.

if "matched_iou" not in globals():
    raise RuntimeError("데이터 셀 아래의 평가·비교 함수 셀을 먼저 실행하세요.")

iou_rows = []
for run_name, metric_model, metric_data, _ in metric_runs:
    iou_result = matched_iou(metric_model, metric_data, IMGSZ, BATCH)
    iou_rows.append(
        {
            "run": run_name,
            "mean_matched_IoU@0.5": iou_result["Mean IoU"],
            "matched": iou_result["Matched"],
            "targets": iou_result["Targets"],
            "predictions": iou_result["Predictions"],
            "unmatched_targets": iou_result["Unmatched targets"],
        }
    )
display(pd.DataFrame(iou_rows).set_index("run").round(4))

In [ ]:
# 10. 재현율: TP / (TP + FN). val 클래스별 값과 클래스 평균을 표시한다.
# Ultralytics의 IoU=0.5, 평균 F1이 최대인 confidence 지점의 값이다.
# IoU 셀의 conf=0.001에서 집계한 매칭 비율과는 기준이 다르다.
recall_rows = []
for run_name, _, _, result in metric_runs:
    class_recall = pd.DataFrame(
        {
            "yolo_idx": result.box.ap_class_index,
            "recall": result.box.r,
        }
    ).merge(class_map, on="yolo_idx", how="left")
    print(run_name)
    display(class_recall.round(4))
    recall_rows.append({"run": run_name, "mean_recall": result.box.mr})
display(pd.DataFrame(recall_rows).set_index("run").round(4))

In [ ]:
# 11. 정밀도: TP / (TP + FP). 재현율과 같은 confidence 지점의 값이다.
precision_rows = []
for run_name, _, _, result in metric_runs:
    class_precision = pd.DataFrame(
        {
            "yolo_idx": result.box.ap_class_index,
            "precision": result.box.p,
        }
    ).merge(class_map, on="yolo_idx", how="left")
    print(run_name)
    display(class_precision.round(4))
    precision_rows.append({"run": run_name, "mean_precision": result.box.mp})
display(pd.DataFrame(precision_rows).set_index("run").round(4))

## 실험 기록

- **실험 이름 / 단계**: (설정 셀 출력의 이름, screen 또는 confirm)
- **이 설정을 고른 이유**:
- **명령**: 실험 준비 셀이 출력한 `uv run python -m models.model_main ...`
- **결과**: screen이면 fold 0 mAP75-95, confirm이면 5 fold 평균 ± 표준편차
- **결론 / 다음 실험**: 

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

from models.train import MAIN_METRIC_KEY

# 12. 앞의 평가 셀 결과를 모아 네 가지 지표를 한 번에 확인한다.
# AP는 주 지표인 클래스 평균 mAP75-95로 표시한다.
# IoU는 IoU >= 0.5로 매칭된 예측만의 평균이다. 미검출 수를 함께 확인한다.
metric_summary = (
    pd.DataFrame(ap_rows)
    .set_index("run")[["mAP75-95"]]
    .join(
        pd.DataFrame(iou_rows).set_index("run")[
            ["mean_matched_IoU@0.5", "matched", "unmatched_targets"]
        ],
        validate="one_to_one",
    )
    .join(pd.DataFrame(recall_rows).set_index("run")[["mean_recall"]], validate="one_to_one")
    .join(pd.DataFrame(precision_rows).set_index("run")[["mean_precision"]], validate="one_to_one")
)
metric_summary = metric_summary[
    [
        "mAP75-95",
        "mean_matched_IoU@0.5",
        "mean_recall",
        "mean_precision",
        "matched",
        "unmatched_targets",
    ]
].rename(columns={"mean_recall": "Recall", "mean_precision": "Precision"})
display(metric_summary.round(4))

# 저장된 에폭별 기록을 읽어 최종 평가표와 학습 추이를 함께 보여 준다.
# 평균 매칭 IoU는 best.pt에서만 계산하므로 에폭별 곡선을 만들지 않는다.

history_paths = sorted(exp_dir.glob("*/results.csv"))
history_paths = [
    path for path in history_paths if int(path.parent.name.rsplit("_fold", 1)[1]) in folds
]
if not history_paths:
    print("에폭별 results.csv가 없습니다. 학습 셀을 먼저 실행하세요.")
else:
    for history_path in history_paths:
        history = pd.read_csv(history_path)
        history.columns = history.columns.str.strip()
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        for loss_name in ("box_loss", "cls_loss", "dfl_loss"):
            for split_name, style in (("train", "-"), ("val", "--")):
                key = f"{split_name}/{loss_name}"
                if key in history:
                    axes[0].plot(
                        history["epoch"],
                        history[key],
                        linestyle=style,
                        marker=".",
                        label=key,
                    )
        for key, label in (
            (MAIN_METRIC_KEY, "mAP75-95"),
            ("metrics/precision(B)", "Precision"),
            ("metrics/recall(B)", "Recall"),
        ):
            if key in history:
                axes[1].plot(history["epoch"], history[key], marker="o", label=label)
        axes[0].set(title="Training / validation loss", ylabel="Loss")
        axes[1].set(title="Validation metrics per epoch", ylabel="Score", ylim=(0, 1))
        for ax in axes:
            ax.set_xlabel("Epoch")
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
            ax.grid(alpha=0.3)
            ax.legend()
        fig.suptitle(f"{exp} / {history_path.parent.name}")
        fig.tight_layout()
        figure_path = history_path.parent / "experiment_history.png"
        fig.savefig(figure_path, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    print("평균 매칭 IoU는 best.pt의 최종 평가표에서 확인하세요.")

In [ ]:
# 13. 30에폭 결과와 기존 스모크를 비교한다. 이 노트북만 최신으로 반영하면 된다.
# Run All에서는 위에서 완료된 같은 30에폭 학습을 건너뛰고 결과를 비교한다.
# 학습이 끊겼다면 이 30에폭 실험의 last.pt에서 이어 간다.
import json
import os
import shlex
from pathlib import Path

import pandas as pd
from IPython.display import display

from models.train import default_lr0, train
from src.config import RUNS_DIR as DEFAULT_RUNS_DIR
from src.config import get_device

TEST_EPOCHS = EPOCHS
TEST_FRACTION = FRACTION
TEST_FOLD = 0
# 비워 두면 현재 RUNS_DIR에서 같은 설정의 완료된 스모크 중 가장 최근 것을 선택한다.
# 직접 지정할 때는 weights/가 들어 있는 run 폴더를 넣는다.
SMOKE_RUN_DIR = None

if "compare_runs" not in globals():
    raise RuntimeError("데이터 셀 아래의 평가·비교 함수 셀을 먼저 실행하세요.")
if TEST_EPOCHS != 30 or TEST_FRACTION != 1.0:
    raise ValueError("이 비교는 EPOCHS = 30, FRACTION = 1.0 설정으로 실행하세요.")
if SMOKE:
    raise RuntimeError("설정 셀을 SMOKE = False로 바꾸고 다시 실행하세요.")
if get_device() != "cuda":
    raise RuntimeError("30에폭 전체 학습은 Colab GPU에서 실행하세요.")

lr0_test = default_lr0(OPTIMIZER) if LR0 is None else LR0
test_runs_dir = Path(RUNS_DIR or DEFAULT_RUNS_DIR)
test_exp = f"{exp_name}_test30"
test_config = {
    "model": MODEL,
    "label": LABEL,
    "split": SPLIT,
    "fold": TEST_FOLD,
    "epochs": TEST_EPOCHS,
    "fraction": TEST_FRACTION,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "aug": AUG,
    "optimizer": OPTIMIZER,
    "lr0": lr0_test,
    "cos_lr": COS_LR,
    "train_kwargs": TRAIN_KWARGS,
}
test_exp_dir = test_runs_dir / test_exp
test_exp_dir.mkdir(parents=True, exist_ok=True)
config_path = test_exp_dir / "test30_config.json"
if config_path.is_file() and json.loads(config_path.read_text()) != test_config:
    raise ValueError("같은 실험 이름의 설정이 달라졌습니다. TAG를 바꿔 새 결과에 저장하세요.")
config_path.write_text(json.dumps(test_config, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame(test_config.items(), columns=["설정", "값"]))

# TRAIN_KWARGS가 비어 있으면 아래 CLI로 똑같이 실행할 수 있다.
test_command = [
    "uv",
    "run",
    "python",
    "-m",
    "models.model_main",
    "--model",
    MODEL,
    "--label",
    LABEL,
    "--split",
    SPLIT,
    "--folds",
    str(TEST_FOLD),
    "--epochs",
    str(TEST_EPOCHS),
    "--fraction",
    str(TEST_FRACTION),
    "--imgsz",
    str(IMGSZ),
    "--batch",
    str(BATCH),
    "--seed",
    str(SEED),
    "--aug",
    AUG,
    "--optimizer",
    OPTIMIZER,
    "--exp",
    test_exp,
    "--runs-dir",
    str(test_runs_dir),
    "--resume",
]
if lr0_test is not None:
    test_command += ["--lr0", str(lr0_test)]
if COS_LR:
    test_command += ["--cos-lr"]
print(f"WANDB_MODE={os.environ.get('WANDB_MODE', 'online')}")
print(shlex.join(test_command))
if TRAIN_KWARGS:
    print(f"CLI 외 추가 train() 인자: {TRAIN_KWARGS}")

# 스모크에서 이어 학습하지 않고 사전학습 가중치로 시작한다.
# 이 셀을 다시 실행하면 이 30에폭 실험의 last.pt에서만 이어 학습한다.
test_save_dir = train(
    MODEL,
    LABEL,
    SPLIT,
    TEST_FOLD,
    epochs=TEST_EPOCHS,
    exp=test_exp,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    fraction=TEST_FRACTION,
    aug=AUG,
    optimizer=OPTIMIZER,
    lr0=lr0_test,
    cos_lr=COS_LR,
    runs_dir=test_runs_dir,
    resume=True,
    **TRAIN_KWARGS,
)

smoke_candidates = find_smoke_runs(test_save_dir)
if SMOKE_RUN_DIR is not None:
    smoke_save_dir = Path(SMOKE_RUN_DIR)
    if smoke_save_dir.resolve() not in {path.resolve() for path in smoke_candidates}:
        raise ValueError(
            "지정한 스모크는 같은 설정의 완료된 5에폭 이하·데이터 10% 결과가 아닙니다."
        )
else:
    smoke_save_dir = smoke_candidates[-1] if smoke_candidates else None
comparison_runs = {}
if smoke_save_dir is not None:
    comparison_runs["Smoke"] = smoke_save_dir
    print(f"비교할 스모크: {smoke_save_dir}")
else:
    print("같은 설정의 완료된 스모크가 없습니다. 기존 runs 폴더를 이 RUNS_DIR에 가져오세요.")
    print("이번에는 30에폭 결과만 표시합니다. 스모크를 가져온 뒤 이 셀을 다시 실행하세요.")
comparison_runs["Test 30"] = test_save_dir
comparison_dir = test_exp_dir / "results" / "smoke_comparison"
comparison = compare_runs(
    comparison_runs,
    LABEL,
    SPLIT,
    TEST_FOLD,
    IMGSZ,
    BATCH,
    comparison_dir,
)
print("best.pt 기준 · 동일한 val 전체 · mAP/재현율/정밀도는 클래스 기준")
print("스모크(데이터 10%)와 30에폭(100%)은 데이터량·학습량이 모두 다른 비교입니다.")
print("평균 IoU는 같은 클래스·IoU >= 0.5 매칭만 포함합니다. 미검출 수도 함께 보세요.")
display(comparison.drop(columns="Weights").round(4))
print(f"비교표·그래프 저장 위치: {comparison_dir}")